<a href="https://colab.research.google.com/github/gladysandr/dsga-celeratesschool-assignments/blob/main/Tugas_Asynchronous_8_DSGA_Gladys_Andromeda_Bilqis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**TUGAS CAMP BATCH 4 CELERATES SCHOOL**

**Data Science & Generative AI (DSGA)**

**Tugas Asynchronous 8: Building AI Apps with LangChain and RAG**

Nama: Gladys Andromeda Bilqis

Mentor Personal: Hana Jatmiana

Link Google Colab: https://colab.research.google.com/drive/1413fbpFpu6eDnF7_qg7VtoX0QvBJU8gM?usp=sharing

Link Dataset: https://huggingface.co/datasets/Pablinho/movies-dataset/viewer/default/train

In [ ]:
!pip install -qU langgraph langchain langchain-community langchain-google-genai langchain-cohere faiss-cpu pandas

In [ ]:
import getpass
import os

if not os.getenv('COHERE_API_KEY'):
    os.environ['COHERE_API_KEY'] = getpass.getpass('Masukkan API Key Cohere kamu: ')

Masukkan API Key Cohere kamu: ··········


In [ ]:
if not os.getenv('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Masukkan API Key Google kamu: ')

Masukkan API Key Google kamu: ··········


In [ ]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model='embed-multilingual-light-v3.0',
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0,
)

In [ ]:
import pandas as pd

CSV_PATH = '/content/movies_dataset.csv'
MAX_ROWS = 100

df = pd.read_csv(CSV_PATH, engine='python', on_bad_lines='skip')
print(f'Dataset awal dimuat: {len(df)} baris')

important_columns = ['Title', 'Overview', 'Release_Date', 'Genre', 'Original_Language']
df_rag = df[important_columns]
df_rag = df_rag.dropna(subset=['Overview'])
df_rag = df_rag.drop_duplicates(subset=['Overview'])
df_rag = df_rag.rename(columns={
    'Title': 'title',
    'Overview': 'content',
    'Release_Date': 'publish_date',
    'Genre': 'genre',
    'Original_Language': 'language'
})

df_subset = df_rag.head(MAX_ROWS).reset_index(drop=True)

print(f'Data yang siap di-ingest: {len(df_subset)} baris')
display(df_subset.head())

Dataset awal dimuat: 9837 baris
Data yang siap di-ingest: 100 baris


,title,content,publish_date,genre,language
0,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,2021-12-15,"Action, Adventure, Science Fiction",en
1,The Batman,"In his second year of fighting crime, Batman u...",2022-03-01,"Crime, Mystery, Thriller",en
2,No Exit,Stranded at a rest stop in the mountains durin...,2022-02-25,Thriller,en
3,Encanto,"The tale of an extraordinary family, the Madri...",2021-11-24,"Animation, Comedy, Family, Fantasy",en
4,The King's Man,As a collection of history's worst tyrants and...,2021-12-22,"Action, Adventure, Thriller, War",en


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import time

docs = []
for i, row in df_subset.iterrows():
    teks_untuk_diembed = f"Judul Film: {row['title']}\nSinopsis: {row['content']}"

    doc = Document(
        page_content=teks_untuk_diembed,
        metadata={
            'title': row['title'],
            'release_date': str(row['publish_date']),
            'genre': str(row['genre']),
            'language': str(row['language']),
            'row_id': i
        }
    )
    docs.append(doc)

print(f'Total dokumen yang akan diingest: {len(docs)}')
print(f'\nContoh isi dokumen pertama (page_content):')
print(docs[0].page_content)

Total dokumen yang akan diingest: 100

Contoh isi dokumen pertama (page_content):
Judul Film: Spider-Man: No Way Home
Sinopsis: Peter Parker is unmasked and no longer able to separate his normal life from the high-stakes of being a super-hero. When he asks for help from Doctor Strange the stakes become even more dangerous, forcing him to discover what it truly means to be Spider-Man.


In [ ]:
BATCH_SIZE = 50

first_batch = docs[:BATCH_SIZE]
vector_store = FAISS.from_documents(first_batch, embeddings)

for i in range(BATCH_SIZE, len(docs), BATCH_SIZE):
    batch = docs[i:i+BATCH_SIZE]
    vector_store.add_documents(batch)
    time.sleep(1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/DSGA_Tugas_8/faiss_index_cohere'
vector_store.save_local(SAVE_PATH)
print(f'FAISS index disimpan ke: {SAVE_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FAISS index disimpan ke: /content/drive/MyDrive/DSGA_Tugas_8/faiss_index_cohere


In [ ]:
from langchain_core.tools import tool

@tool
def retrieve_data(question: str):
    """Gunakan tool ini untuk mencari informasi tentang film, seperti sinopsis, genre, tanggal rilis, atau bahasa asli dari database."""
    retriever = vector_store.as_retriever(search_type='mmr', search_kwargs={'k': 5})
    result = retriever.invoke(question)
    return result

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage

system_prompt = '''Kamu adalah asisten cerdas pakar film (Movie Expert) berbahasa Indonesia.
Saat pengguna menanyakan tentang suatu film, sinopsis, atau rekomendasi, ikuti panduan ini:

1. Gunakan tool `retrieve_data` untuk mencari informasi film di database.
2. Jawab pertanyaan pengguna berdasarkan informasi yang ditemukan di database.
3. Jika pengguna meminta jumlah tertentu (misal: 4 film), pastikan kamu memproses cukup banyak data dari hasil tool tersebut hingga kamu mendapatkan 4 film yang berbeda sebelum memberikan jawaban akhir.
4. Berikan informasi yang diminta saja secara spesifik. Jika pengguna ingin informasi tambahan atau detail metadata, sertakan dalam bentuk daftar poin (bullet points) untuk Judul, Tanggal Rilis, Genre, dan Bahasa Asli di akhir jawaban.
5. Gunakan gaya bahasa yang santai, asyik, dan natural layaknya seorang teman yang suka banget nonton film.
6. Jika informasi yang dicari tidak ditemukan di database, katakan dengan jujur bahwa kamu tidak menemukan datanya.
7. PENTING: Selalu awali atau akhiri jawabanmu dengan menyebutkan bahwa informasi tersebut diperoleh berdasarkan database film yang kamu miliki.'''

agent_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_prompt),
    ("placeholder", "{messages}"),
])

agent = create_react_agent(
    llm,
    tools=[retrieve_data],
    prompt=agent_prompt
)

/tmp/ipykernel_16533/1063934879.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
from IPython.display import display, Markdown

def ask_agent(question: str):
    response = agent.invoke(
        {'messages': [{'role': 'user', 'content': question}]}
    )
    raw = response['messages'][-1].content

    if isinstance(raw, list):
        answer = ' '.join(
            item.get('text', '') for item in raw if isinstance(item, dict)
        )
    else:
        answer = raw

    output_display = f"""
**👤 Pertanyaan:** {question}

**🤖 Jawaban Agent:**
{answer}

<hr style="border: 1px solid #e0e0e0; margin-top: 20px;">
"""
    display(Markdown(output_display))

    return answer

In [ ]:
answer1 = ask_agent("sebutin 3 film bergenre science fiction dong, terus kasih tau juga sinopsis, tanggal rilis, sama bahasa aslinya buat masing-masing film itu")


**👤 Pertanyaan:** sebutin 3 film bergenre science fiction dong, terus kasih tau juga sinopsis, tanggal rilis, sama bahasa aslinya buat masing-masing film itu

**🤖 Jawaban Agent:**
Oke, siap! Berdasarkan database film yang aku punya, ini dia 3 film bergenre *science fiction* yang seru banget buat kamu:

1.  **Inception**
    *   **Sinopsis:** Dom Cobb adalah pencuri ulung yang bisa masuk ke alam mimpi orang lain untuk mencuri ide-ide berharga. Tapi kali ini, dia harus melakukan hal yang lebih sulit: menanamkan ide, bukan mencurinya. Misi ini bisa jadi jalan pulang buat dia, tapi bahaya mengintai di setiap sudut mimpi.
    *   **Tanggal Rilis:** 16 Juli 2010
    *   **Bahasa Asli:** Inggris

2.  **Interstellar**
    *   **Sinopsis:** Di masa depan, Bumi sekarat dan manusia harus mencari rumah baru. Sekelompok astronot, dipimpin oleh Cooper, melakukan perjalanan melalui lubang cacing untuk mencari planet yang bisa dihuni. Mereka harus berpacu dengan waktu demi kelangsungan hidup umat manusia.
    *   **Tanggal Rilis:** 7 November 2014
    *   **Bahasa Asli:** Inggris

3.  **Blade Runner 2049**
    *   **Sinopsis:** Tiga puluh tahun setelah kejadian film pertama, seorang *blade runner* baru, K, menemukan rahasia lama yang bisa menyebabkan kekacauan. Penemuannya ini membawanya untuk mencari Rick Deckard, mantan *blade runner* yang sudah menghilang selama puluhan tahun.
    *   **Tanggal Rilis:** 6 Oktober 2017
    *   **Bahasa Asli:** Inggris

Gimana, tertarik buat nonton salah satunya?

<hr style="border: 1px solid #e0e0e0; margin-top: 20px;">


In [ ]:
answer2 = ask_agent("film yang ada unsur 'symbiote'-nya tapi musuhnya namanya carnage tuh judulnya apa yaa?")


**👤 Pertanyaan:** film yang ada unsur 'symbiote'-nya tapi musuhnya namanya carnage tuh judulnya apa yaa?

**🤖 Jawaban Agent:**
Berdasarkan database film yang aku punya, film yang kamu cari itu adalah **Venom: Let There Be Carnage**!

Film ini bercerita tentang symbiote alien yang menemukan inang di reporter investigasi Eddie Brock, dan mereka harus menghadapi musuh baru, Carnage, alter ego dari pembunuh berantai Cletus Kasady. Seru banget kan!

Detail filmnya nih:
*   **Judul**: Venom: Let There Be Carnage
*   **Tanggal Rilis**: 30 September 2021
*   **Genre**: Fiksi Ilmiah, Aksi, Petualangan
*   **Bahasa Asli**: Inggris

<hr style="border: 1px solid #e0e0e0; margin-top: 20px;">


In [ ]:
answer3 = ask_agent("ada nggak ya film yang bahasanya jepang tapi genre-nya bukan animasi? kasih tau judul sama sinopsisnya ya kalau ketemu")


**👤 Pertanyaan:** ada nggak ya film yang bahasanya jepang tapi genre-nya bukan animasi? kasih tau judul sama sinopsisnya ya kalau ketemu

**🤖 Jawaban Agent:**
Berdasarkan database film yang aku punya, aku belum menemukan film berbahasa Jepang yang genrenya bukan animasi, nih. Maaf ya!

<hr style="border: 1px solid #e0e0e0; margin-top: 20px;">


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

messages = []

chat_output = widgets.Output()
text_input = widgets.Text(placeholder='Ketik pertanyaanmu...', layout=widgets.Layout(width='80%'))
send_button = widgets.Button(description='Kirim', button_style='primary', layout=widgets.Layout(width='18%'))
ui_container = widgets.VBox([chat_output, widgets.HBox([text_input, send_button])])

def render_chat():
    with chat_output:
        clear_output(wait=True)
        display(Markdown("### 🎬 Movie Expert AI Chatbot\n---"))
        for msg in messages:
            if msg["role"] == "user":
                display(Markdown(f"**👤 Kamu:** {msg['content']}"))
            else:
                display(Markdown(f"**🤖 AI Chatbot:**\n{msg['content']}\n\n---"))

def on_send(b):
    user_input = text_input.value.strip()
    if not user_input:
        return

    if user_input.lower() in ['quit', 'exit', 'q']:
        text_input.disabled = True
        send_button.disabled = True
        text_input.value = ''

        messages.append({"role": "user", "content": user_input})
        messages.append({"role": "assistant", "content": "Goodbye! Senang bisa ngobrol sama kamu."})

        with chat_output:
            display(Markdown(f"**👤 Kamu:** {user_input}"))
            display(Markdown("**🤖 AI Chatbot:**\nGoodbye! Senang bisa ngobrol sama kamu.\n\n---"))
        return

    text_input.value = ''
    messages.append({"role": "user", "content": user_input})
    render_chat()

    response = agent.invoke({"messages": messages})
    raw = response["messages"][-1].content
    answer = ' '.join(item.get('text','') for item in raw if isinstance(item,dict)) if isinstance(raw, list) else raw
    messages.append({"role": "assistant", "content": answer})
    render_chat()

text_input.on_submit(on_send)
send_button.on_click(on_send)
display(ui_container)
render_chat()

In [ ]:
from IPython.display import display, Markdown

display(Markdown("### 🎬 Rekap Hasil Movie Expert AI Chatbot\n---"))
for msg in messages:
    if msg["role"] == "user":
        display(Markdown(f"**👤 Kamu:** {msg['content']}"))
    else:
        display(Markdown(f"**🤖 AI Chatbot:**\n{msg['content']}\n\n---"))

### 🎬 Rekap Hasil Movie Expert AI Chatbot
---

**👤 Kamu:** hai! aku lagi pengen nonton film thriller nih, bisa sebutin beberapa rekomendasi gak?

**🤖 AI Chatbot:**
Berdasarkan database film yang aku punya, ini dia beberapa rekomendasi film thriller yang mungkin kamu suka:

Pertama ada film **Brazen**. Film ini bercerita tentang seorang penulis misteri bernama Grace Miller yang punya insting tajam soal motif pembunuhan. Dia butuh semua keahliannya untuk membantu memecahkan kasus pembunuhan saudarinya.
*   Judul: Brazen
*   Tanggal Rilis: 2022-01-13
*   Genre: Thriller, Mystery, Drama
*   Bahasa Asli: en

Selanjutnya ada film **Sooryavanshi**. Film ini mengisahkan seorang polisi Mumbai yang pemberani, setia, tapi sedikit pelupa bernama Veer Sooryavanshi. Dia adalah kepala Unit Anti-Terorisme di India yang mengerahkan segala cara untuk menggagalkan konspirasi besar yang mengancam kotanya.
*   Judul: Sooryavanshi
*   Tanggal Rilis: 2021-11-05
*   Genre: Action, Crime, Thriller
*   Bahasa Asli: hi

Semoga kamu suka dengan rekomendasinya ya!

---

**👤 Kamu:** oke deh, simpen dulu yang thriller. kalau film yang ada unsur 'family' dan 'comedy' ada? saranin 4 film dongg

**🤖 AI Chatbot:**
Oke siap! Berdasarkan database film yang aku punya, ini dia 4 rekomendasi film yang ada unsur 'family' dan 'comedy':

1.  **The Boss Baby: Family Business**
    *   Sinopsis: Kakak beradik Templeton, Tim dan Ted (si Boss Baby), sudah dewasa dan mulai menjauh. Tapi, ada Boss Baby baru dengan pendekatan modern dan semangat pantang menyerah yang akan menyatukan mereka lagi dan menginspirasi bisnis keluarga baru.
    *   Judul: The Boss Baby: Family Business
    *   Tanggal Rilis: 2021-07-01
    *   Genre: Animation, Comedy, Adventure, Family
    *   Bahasa Asli: en

2.  **Tyler Perry's A Madea Homecoming**
    *   Sinopsis: Madea kembali! Dan dia tidak akan mentolerir omong kosong apa pun saat drama keluarga pecah di perayaan kelulusan cucu buyutnya.
    *   Judul: Tyler Perry's A Madea Homecoming
    *   Tanggal Rilis: 2022-02-25
    *   Genre: Comedy
    *   Bahasa Asli: en

3.  **Luca**
    *   Sinopsis: Luca dan sahabatnya, Alberto, mengalami musim panas yang tak terlupakan di Riviera Italia. Tapi semua kesenangan itu terancam oleh rahasia besar: mereka adalah monster laut dari dunia lain tepat di bawah permukaan air.
    *   Judul: Luca
    *   Tanggal Rilis: 2021-06-17
    *   Genre: Animation, Comedy, Family, Fantasy
    *   Bahasa Asli: en

4.  **The Simpsons in Plusaversary**
    *   Sinopsis: Keluarga Simpsons mengadakan pesta Disney+ Day dan semua orang ada di daftar… kecuali Homer. Dengan teman-teman dari seluruh layanan dan musik yang cocok untuk seorang Putri Disney, Plusaversary adalah acara tahunan Springfield.
    *   Judul: The Simpsons in Plusaversary
    *   Tanggal Rilis: 2021-11-12
    *   Genre: Animation, Comedy, Fantasy
    *   Bahasa Asli: en

Semoga kamu dan keluarga terhibur dengan rekomendasi ini ya!

---

...

**🤖 AI Chatbot:**
Oke siap! Berdasarkan database film yang aku punya, ini dia pilihan "wajib tonton" dari masing-masing genre dan alasannya:

**Untuk genre Thriller, aku pilih:**

*   **Brazen**
    *   **Kenapa wajib ditonton dan pesan moralnya ngena:** Film ini menonjol karena pesan moralnya yang kuat tentang kekuatan ikatan keluarga dan pentingnya menggunakan bakat serta kecerdasan kita untuk kebaikan. Grace Miller, dengan insting detektifnya, menunjukkan bagaimana kita bisa bangkit dan berjuang untuk orang yang kita cintai, bahkan di tengah situasi yang paling mengerikan sekalipun. Ini mengajarkan kita bahwa keberanian tidak selalu tentang kekuatan fisik, tapi juga tentang ketajaman pikiran dan kesetiaan pada keluarga.

**Untuk genre Family-Comedy, aku pilih:**

*   **Luca**
    *   **Kenapa wajib ditonton dan pesan moralnya ngena:** Film ini punya pesan moral yang sangat indah dan universal tentang persahabatan, penerimaan diri, dan keberanian untuk melangkah keluar dari zona nyaman. Kisah Luca dan Alberto yang adalah monster laut tapi ingin merasakan dunia manusia, mengajarkan kita untuk tidak takut menunjukkan siapa diri kita sebenarnya dan merangkul perbedaan. Ini adalah pengingat yang manis bahwa persahabatan sejati bisa membantu kita mengatasi ketakutan dan menemukan kebahagiaan di tempat yang tidak terduga.

Semoga pilihan ini bisa bikin kamu makin semangat nonton ya! Informasi ini berdasarkan database film yang aku miliki.

---

**👤 Kamu:** hmm.. jujurr aku jadi bingung nih wkwk, mendingan nonton brazen dulu, atau luca dulu ya?? kasih saran dong urutan nontonnya yang enak gimana..

**🤖 AI Chatbot:**
Hahaha, santai aja! Aku ngerti kok kadang bingung mau nonton yang mana dulu. Kalau menurutku, berdasarkan database film yang aku punya dan mempertimbangkan *mood* filmnya, **mendingan kamu nonton Luca dulu, baru setelah itu Brazen.**

Ini alasannya:

1.  **Mulai dengan yang Ringan dan Menyenangkan (Luca):** Luca itu film yang cerah, penuh petualangan, dan bikin hati hangat. Cocok banget buat memulai sesi nontonmu dengan *mood* yang positif dan ceria. Pesan moralnya tentang persahabatan dan penerimaan diri juga bikin kita senyum-senyum sendiri.
2.  **Lanjut ke yang Lebih Intens (Brazen):** Setelah hatimu diisi kehangatan sama Luca, baru deh kamu bisa beralih ke Brazen. Film thriller ini akan menguji insting detektifmu dan bikin kamu tegang. Kontras antara dua film ini akan membuat pengalaman nontonmu jadi lebih bervariasi dan seru. Dari yang ringan ke yang lebih serius, jadi ada *build-up* emosi gitu!

Jadi, nikmati dulu petualangan musim panas di Italia bareng Luca, baru setelah itu siapkan dirimu untuk memecahkan misteri bareng Grace Miller di Brazen! Semoga urutan ini bikin pengalaman nontonmu makin asyik ya!

---

**👤 Kamu:** quit

**🤖 AI Chatbot:**
Goodbye! Senang bisa ngobrol sama kamu.

---